# W2 D1 — Alert Correlation Pipeline

**Scenario:** GeekShop e-commerce platform — `payment-svc` DB connection pool exhaustion  
**Goal:** Correlate 20 raw alerts → deduplicated clusters using topology-aware fingerprinting

## Pipeline overview
1. Load topology graph (`services.json`) + raw alerts (`alerts_sample.jsonl`)
2. Build undirected adjacency map (1-hop neighbors)
3. Generate fingerprints: `service|metric|severity` (no timestamp, no value)
4. Deduplicate by fingerprint → candidate set
5. Temporal window grouping (`gap_sec = 300`)
6. Topology expansion (`max_hop = 1`) — merge windows whose **services** are within 1 hop
7. Output `results/cluster_summary.json`

In [8]:
# Cell 1 — Imports & configuration
import json
import os
from datetime import datetime, timezone
from collections import defaultdict
from pathlib import Path

# ── Tuneable hyperparameters ──────────────────────────────────────────────────
GAP_SEC  = 300   # max seconds between consecutive alerts to stay in same window
MAX_HOP  = 1     # topology distance for merging cross-service windows
DATA_DIR = Path('dataset')
OUT_DIR  = Path('results')
OUT_DIR.mkdir(exist_ok=True)

print(f'Config: GAP_SEC={GAP_SEC}, MAX_HOP={MAX_HOP}')
print(f'Output directory: {OUT_DIR.resolve()}')

Config: GAP_SEC=300, MAX_HOP=1
Output directory: C:\Users\ASUS\Documents\AIOps\aiops-nguyenduchao\w2\d1\results


In [9]:
# Cell 2 — Load data

def load_services(path: Path) -> dict:
    """Load services.json and return the raw dict."""
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def load_alerts(path: Path) -> list[dict]:
    """Load alerts_sample.jsonl → list of alert dicts, sorted by timestamp."""
    alerts = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                alerts.append(json.loads(line))
    # Ensure chronological order
    alerts.sort(key=lambda a: a['ts'])
    return alerts

services_data = load_services(DATA_DIR / 'services.json')
alerts        = load_alerts(DATA_DIR / 'alerts_sample.jsonl')

print(f'Loaded {len(services_data["services"])} services, {len(services_data["stores"])} stores, {len(services_data["edges"])} edges')
print(f'Loaded {len(alerts)} alerts')
print('\nFirst 3 alerts:')
for a in alerts[:3]:
    print(f'  [{a["id"]}] {a["ts"]} | {a["service"]} | {a["metric"]} | {a["severity"]}')

Loaded 10 services, 4 stores, 17 edges
Loaded 20 alerts

First 3 alerts:
  [a-0001] 2026-06-12T09:42:01Z | payment-svc | db_connection_pool_used_ratio | warn
  [a-0002] 2026-06-12T09:42:18Z | payment-svc | db_connection_pool_used_ratio | crit
  [a-0003] 2026-06-12T09:42:22Z | payment-svc | latency_p99_ms | crit


In [10]:
# Cell 3 — Build topology adjacency map

def build_adjacency(services_data: dict) -> dict[str, set]:
    """
    Build undirected adjacency map from services.json edges.
    Includes both service→service and service→store hops.
    Returns: { service_name: {neighbor1, neighbor2, ...} }
    """
    adj: dict[str, set] = defaultdict(set)
    for edge in services_data['edges']:
        src, dst = edge['from'], edge['to']
        adj[src].add(dst)
        adj[dst].add(src)   # undirected — propagation can go both ways
    return dict(adj)

def get_neighbors_within_hops(service: str, adj: dict, max_hop: int) -> set:
    """
    BFS up to max_hop hops from a service.
    Returns set of reachable nodes (excluding the start node itself).
    """
    visited = {service}
    frontier = {service}
    for _ in range(max_hop):
        next_frontier = set()
        for node in frontier:
            for nb in adj.get(node, set()):
                if nb not in visited:
                    next_frontier.add(nb)
                    visited.add(nb)
        frontier = next_frontier
    return visited - {service}

adjacency = build_adjacency(services_data)

print('Adjacency map (service → neighbors):')
for svc, neighbors in sorted(adjacency.items()):
    # Only print service nodes (not store nodes) for readability
    svc_names = {s['name'] for s in services_data['services']}
    if svc in svc_names:
        print(f'  {svc:20s} → {sorted(neighbors)}')

Adjacency map (service → neighbors):
  auth-svc             → ['edge-lb']
  cart-svc             → ['cart-redis', 'catalog-svc', 'checkout-svc']
  catalog-svc          → ['cart-svc', 'catalog-db', 'edge-lb', 'recommender-svc']
  checkout-svc         → ['cart-svc', 'edge-lb', 'inventory-svc', 'notification-svc', 'payment-svc']
  edge-lb              → ['auth-svc', 'catalog-svc', 'checkout-svc', 'search-svc']
  inventory-svc        → ['catalog-db', 'checkout-svc']
  notification-svc     → ['checkout-svc', 'kafka-events']
  payment-svc          → ['checkout-svc', 'payments-db']
  recommender-svc      → ['catalog-db', 'catalog-svc']
  search-svc           → ['catalog-db', 'edge-lb']


In [11]:
# Cell 4 — Fingerprinting & deduplication

def make_fingerprint(alert: dict) -> str:
    """
    Fingerprint = service|metric|severity
    Intentionally excludes timestamp and value so that repeated firing
    of the same condition (e.g. payment-svc latency_p99_ms crit fired 3×)
    is recognised as the SAME symptom, not a new alert.
    """
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

# Attach fingerprint to every alert
for a in alerts:
    a['fingerprint'] = make_fingerprint(a)

# Show duplication stats
fp_counts = defaultdict(list)
for a in alerts:
    fp_counts[a['fingerprint']].append(a['id'])

print('Fingerprint → alert IDs (showing duplicates only):')
for fp, ids in sorted(fp_counts.items()):
    if len(ids) > 1:
        print(f'  {fp}')
        print(f'    → {ids}  ({len(ids)} occurrences — all but first are duplicates)')

unique_fps = set(a['fingerprint'] for a in alerts)
print(f'\nTotal alerts : {len(alerts)}')
print(f'Unique fingerprints: {len(unique_fps)}')
print(f'Duplicate alerts   : {len(alerts) - len(unique_fps)}')

Fingerprint → alert IDs (showing duplicates only):
  payment-svc|db_connection_pool_used_ratio|crit
    → ['a-0002', 'a-0011']  (2 occurrences — all but first are duplicates)
  payment-svc|latency_p99_ms|crit
    → ['a-0003', 'a-0008', 'a-0015']  (3 occurrences — all but first are duplicates)

Total alerts : 20
Unique fingerprints: 17
Duplicate alerts   : 3


In [12]:
# Cell 5 — Core correlate() pipeline

def ts_to_epoch(ts_str: str) -> float:
    """Parse ISO-8601 UTC timestamp to Unix epoch seconds."""
    dt = datetime.fromisoformat(ts_str.replace('Z', '+00:00'))
    return dt.timestamp()

def are_topologically_related(services_a: set, services_b: set,
                               adj: dict, max_hop: int) -> bool:
    """
    Return True if ANY service in group A is within max_hop hops
    of ANY service in group B in the topology graph.
    """
    for svc in services_a:
        reachable = get_neighbors_within_hops(svc, adj, max_hop)
        reachable.add(svc)   # include self
        if reachable & services_b:   # non-empty intersection
            return True
    return False


def correlate(alerts: list[dict],
              adj: dict,
              gap_sec: int = GAP_SEC,
              max_hop: int = MAX_HOP) -> list[dict]:
    """
    Two-phase alert correlation:

    Phase 1 — Temporal deduplication
        Walk alerts sorted by time.  Open a new window when the gap
        from the PREVIOUS alert in the same fingerprint group exceeds gap_sec.
        Within a window, only the FIRST occurrence of each fingerprint is kept;
        subsequent fires are counted as duplicates and suppressed.

    Phase 2 — Topology merge
        After Phase 1 we have a list of candidate windows.  Merge any two
        windows whose time ranges OVERLAP or are within gap_sec of each other
        AND whose service sets are topologically adjacent (within max_hop hops).
        Repeat until no more merges are possible.

    Returns a list of cluster dicts ready for JSON serialisation.
    """
    # ── Phase 1: per-fingerprint temporal windows ──────────────────────────
    # Group alerts by fingerprint, track last-seen epoch per fingerprint
    fp_last_ts: dict[str, float] = {}   # fingerprint → epoch of last kept alert
    windows: list[dict] = []            # list of open windows
    # We also need to track which window each kept alert belongs to.
    # Strategy: one window per (fingerprint, burst-index).
    # For simplicity in Phase 2 we'll later group by time overlap.

    fp_windows: dict[str, dict] = {}   # fingerprint → current open window

    for alert in alerts:   # already sorted by ts
        fp  = alert['fingerprint']
        ts  = ts_to_epoch(alert['ts'])

        if fp not in fp_windows:
            # Start first window for this fingerprint
            new_win = {
                '_alerts': [alert],
                '_fps': {fp},
                'services': {alert['service']},
                't_start': ts,
                't_end': ts,
            }
            fp_windows[fp] = new_win
            windows.append(new_win)
        else:
            last_ts = fp_windows[fp]['t_end']
            if ts - last_ts <= gap_sec:
                # Same burst — duplicate, suppress (update end time)
                fp_windows[fp]['t_end'] = ts
                fp_windows[fp]['_alerts'].append(alert)
                # Mark as dup (don't count toward unique fingerprint set)
            else:
                # Gap too large — start a NEW window for this fingerprint
                new_win = {
                    '_alerts': [alert],
                    '_fps': {fp},
                    'services': {alert['service']},
                    't_start': ts,
                    't_end': ts,
                }
                fp_windows[fp] = new_win
                windows.append(new_win)

    # ── Phase 2: topology-aware window merge ───────────────────────────────
    # Iteratively merge pairs of windows that are:
    #   (a) temporally overlapping OR within gap_sec of each other, AND
    #   (b) topologically adjacent (within max_hop hops)

    def windows_are_close(w1, w2, gap_sec):
        """True if the two windows' time ranges overlap or are within gap_sec."""
        return not (w1['t_end'] + gap_sec < w2['t_start'] or
                    w2['t_end'] + gap_sec < w1['t_start'])

    merged = True
    while merged:
        merged = False
        new_windows = []
        used = [False] * len(windows)
        for i in range(len(windows)):
            if used[i]:
                continue
            base = windows[i]
            for j in range(i + 1, len(windows)):
                if used[j]:
                    continue
                cand = windows[j]
                if (windows_are_close(base, cand, gap_sec) and
                        are_topologically_related(base['services'],
                                                   cand['services'],
                                                   adj, max_hop)):
                    # Merge cand into base
                    base['_alerts'].extend(cand['_alerts'])
                    base['_fps']    |= cand['_fps']
                    base['services']|= cand['services']
                    base['t_start'] = min(base['t_start'], cand['t_start'])
                    base['t_end']   = max(base['t_end'],   cand['t_end'])
                    used[j] = True
                    merged  = True
            new_windows.append(base)
        windows = new_windows

    return windows


windows = correlate(alerts, adjacency, gap_sec=GAP_SEC, max_hop=MAX_HOP)
print(f'Phase 1+2 complete → {len(windows)} cluster(s) from {len(alerts)} alerts')

Phase 1+2 complete → 2 cluster(s) from 20 alerts


In [13]:
# Cell 6 — Build cluster_summary.json and write to disk

from datetime import timezone

def epoch_to_iso(epoch: float) -> str:
    return datetime.fromtimestamp(epoch, tz=timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def build_summary(alerts_raw: list[dict], windows: list[dict]) -> dict:
    """Convert internal window list to the required cluster_summary format."""
    # Sort windows by start time
    windows_sorted = sorted(windows, key=lambda w: w['t_start'])

    clusters = []
    for idx, win in enumerate(windows_sorted):
        cluster_id = f'c-{idx+1:03d}-{idx:03d}'

        # All alerts whose fingerprint appears in this window
        win_fps = win['_fps']
        member_alerts = [a for a in win['_alerts']]

        # Severity ordering
        sev_order = {'info': 0, 'warn': 1, 'crit': 2}
        max_sev = max(member_alerts, key=lambda a: sev_order.get(a['severity'], 0))['severity']

        clusters.append({
            'cluster_id':    cluster_id,
            'alert_count':   len(member_alerts),
            'services':      sorted(win['services']),
            'time_range':    [epoch_to_iso(win['t_start']), epoch_to_iso(win['t_end'])],
            'max_severity':  max_sev,
            'fingerprints':  sorted(win['_fps']),
        })

    input_alerts   = len(alerts_raw)
    output_clusters= len(clusters)
    reduction_ratio= round(1 - output_clusters / input_alerts, 4)

    return {
        'input_alerts':    input_alerts,
        'output_clusters': output_clusters,
        'reduction_ratio': reduction_ratio,
        'clusters':        clusters,
    }


summary = build_summary(alerts, windows)

out_path = OUT_DIR / 'cluster_summary.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Written: {out_path}')
print(f'input_alerts    : {summary["input_alerts"]}')
print(f'output_clusters : {summary["output_clusters"]}')
print(f'reduction_ratio : {summary["reduction_ratio"]}  (must be >= 0.5)')
print()
for c in summary['clusters']:
    print(f'  [{c["cluster_id"]}]  {c["alert_count"]} alerts | max_sev={c["max_severity"]} | services={c["services"]}')
    print(f'     time: {c["time_range"][0]} → {c["time_range"][1]}')
    print(f'     fps : {c["fingerprints"]}')

Written: results\cluster_summary.json
input_alerts    : 20
output_clusters : 2
reduction_ratio : 0.9  (must be >= 0.5)

  [c-001-000]  19 alerts | max_sev=crit | services=['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'search-svc']
     time: 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
     fps : ['cart-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'checkout-svc|latency_p99_ms|crit', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'edge-lb|upstream_5xx_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'notification-svc|queue_depth|crit', 'notification-svc|queue_lag_ms|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|db_connection_pool_used_ratio|warn', 'payment-svc|error_rate|crit', 'payment-svc|error_rate|warn', 'payment-svc|latency_p99_ms|crit', 'search-svc|catalog_db_query_time_ms|warn']
  [c-002-001]  1 alerts | max_sev=warn | services=['recommender-

In [14]:
# Cell 7 — Validate output

def validate_summary(summary: dict) -> None:
    """Acceptance-criteria checks — raises AssertionError on failure."""
    assert summary['input_alerts'] > 0, 'input_alerts must be > 0'
    assert summary['output_clusters'] >= 1, 'Must have at least 1 cluster'
    assert summary['reduction_ratio'] >= 0.5, (
        f'reduction_ratio {summary["reduction_ratio"]} < 0.5'
    )
    # Verify formula
    expected_ratio = round(1 - summary['output_clusters'] / summary['input_alerts'], 4)
    assert summary['reduction_ratio'] == expected_ratio, 'reduction_ratio formula mismatch'

    for c in summary['clusters']:
        assert 'services' in c and len(c['services']) > 0, \
            f'Cluster {c["cluster_id"]} missing services'
        assert 'time_range' in c and len(c['time_range']) == 2, \
            f'Cluster {c["cluster_id"]} missing time_range'
        assert 'fingerprints' in c and len(c['fingerprints']) > 0, \
            f'Cluster {c["cluster_id"]} missing fingerprints'

    print('All validation checks PASSED')

# Re-load from disk to verify JSON round-trip
with open(OUT_DIR / 'cluster_summary.json', encoding='utf-8') as f:
    loaded = json.load(f)

validate_summary(loaded)
print()
print('Full cluster_summary.json content:')
print(json.dumps(loaded, indent=2))

All validation checks PASSED

Full cluster_summary.json content:
{
  "input_alerts": 20,
  "output_clusters": 2,
  "reduction_ratio": 0.9,
  "clusters": [
    {
      "cluster_id": "c-001-000",
      "alert_count": 19,
      "services": [
        "cart-svc",
        "checkout-svc",
        "edge-lb",
        "notification-svc",
        "payment-svc",
        "search-svc"
      ],
      "time_range": [
        "2026-06-12T09:42:01Z",
        "2026-06-12T09:48:30Z"
      ],
      "max_severity": "crit",
      "fingerprints": [
        "cart-svc|latency_p99_ms|warn",
        "checkout-svc|downstream_payment_error_rate|crit",
        "checkout-svc|latency_p99_ms|crit",
        "checkout-svc|latency_p99_ms|warn",
        "checkout-svc|request_drop_rate|crit",
        "edge-lb|p99_latency_ms|crit",
        "edge-lb|upstream_5xx_rate|crit",
        "edge-lb|upstream_5xx_rate|warn",
        "notification-svc|queue_depth|crit",
        "notification-svc|queue_lag_ms|warn",
        "payment-svc|